# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the data contract for the **Refresh / Content Opportunity Scoring** lane, using the full warehouse release.

## 1. Unit of analysis + time window

**1. Unit of Analysis:** One row = One unique content item per client (`client_hash_id` + `content_hash_id`) aggregated at the **monthly** level.

**2. Tables:** 
- `fact_content_daily_performance` (metrics: impressions, clicks, position)
- `dim_content` (metadata: word_count, content_type, main_intent)
- `dim_clients` (context: history coverage)

**3. Time Window:** 
- **Feature Window:** February 2026 (`month='2026-02'`).
- **Label Window:** March 2026 (`month='2026-03'`).

**4. Label/Proxy:** `is_declining` — A boolean flag indicating if March clicks dropped below 80% of February clicks (representing a >20% decline).

**5. Deliberately Excluded:** `trend_direction` and `trend_pct` from the warehouse tables. These are derived from the very performance we are predicting and would cause immediate data leakage.

In [1]:
import duckdb
import os, getpass
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Setup connection
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Contract setup complete. Ready to verify.")

Contract setup complete. Ready to verify.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Logic / Why |
|---|---|---|
| **Feature** | `clicks_feb`, `imps_feb`, `pos_feb`, `word_count`, `intent` | Knowable before March 1st. |
| **Label** | `is_declining` | Calculated outcome: `clicks_march < 0.8 * clicks_feb`. |
| **Context** | `client_hash_id`, `content_hash_id` | Used for joining and grouping, not for training. |
| **Excluded** | `trend_direction` | Leakage: it is computed from the outcome window. |
| **Excluded** | `clicks_march` | Excluded from features because it defines the label. |

In [2]:
# Quick check of the fact table schema
con.sql(f"SELECT * FROM {FACT} LIMIT 0").df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

## 3. Verify it with queries

### Query 1: The Grain
Proving that the daily performance table follows the `(report_date, client, content)` grain (no duplicates per day).

In [3]:
grain_violation = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*)
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Grain violations found: {len(grain_violation)}")

Grain violations found: 0


### Query 2: Slice Counts and Span
Verifying the row count and date range for our mid-panel month.

In [4]:
stats = con.sql(f"""
    SELECT 
        COUNT(*) as total_daily_rows,
        COUNT(DISTINCT content_hash_id) as unique_content_items,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date
    FROM {FACT}
    WHERE month = '2026-03'
""").df()
stats

,total_daily_rows,unique_content_items,first_date,last_date
0,9841378,331437,2026-03-01,2026-03-31


### Query 3: Availability (IS TRUE)
Checking how many rows have valid GA4 data available vs. zero-filled/missing.

In [5]:
availability = con.sql(f"""
    SELECT 
        ga4_data_available IS TRUE as ga4_ok,
        COUNT(*) as rows
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY 1
""").df()
availability

,ga4_ok,rows
0,False,9427412
1,True,413966


## 4. Five features + The Trap

We build a small feature frame and demonstrate the leakage trap.

In [6]:
# 1. Build the aggregated dataset
df_contract = con.sql(f"""
    SELECT 
        f.client_hash_id, 
        f.content_hash_id,
        -- Features (Knowable at decision moment)
        SUM(CASE WHEN month = '2026-02' THEN gsc_clicks ELSE 0 END) as clicks_feb,
        SUM(CASE WHEN month = '2026-02' THEN gsc_impressions ELSE 0 END) as imps_feb,
        AVG(CASE WHEN month = '2026-02' THEN gsc_avg_position END) as pos_feb,
        ANY_VALUE(c.word_count) as word_count,
        ANY_VALUE(c.main_intent) as intent,
        
        -- Target components
        SUM(CASE WHEN month = '2026-03' THEN gsc_clicks ELSE 0 END) as clicks_march,
        
        -- THE TRAP: Future impressions (Leakage)
        SUM(CASE WHEN month = '2026-03' THEN gsc_impressions ELSE 0 END) as imps_march_LEAK
        
    FROM {FACT} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE month IN ('2026-02', '2026-03')
    GROUP BY 1, 2
    HAVING clicks_feb > 5 -- Volume floor for stability
""").df()

df_contract['is_declining'] = (df_contract['clicks_march'] < 0.8 * df_contract['clicks_feb']).astype(int)

# 2. The Experiment
model = RandomForestClassifier(n_estimators=50, random_state=42)
X = df_contract.fillna(0)
y = df_contract['is_declining']

features_honest = ['clicks_feb', 'imps_feb', 'pos_feb']
features_leaky = features_honest + ['imps_march_LEAK']

model.fit(X[features_honest], y)
print(f"Honest ROC-AUC: {roc_auc_score(y, model.predict_proba(X[features_honest])[:,1]):.3f}")

model.fit(X[features_leaky], y)
print(f"Leaky ROC-AUC: {roc_auc_score(y, model.predict_proba(X[features_leaky])[:,1]):.3f} (The Trap!)")

# 3. Remove the trap
df_contract = df_contract.drop(columns=['imps_march_LEAK'])
print("Leaky column removed. Contract is honest.")

Honest ROC-AUC: 1.000
Leaky ROC-AUC: 1.000 (The Trap!)
Leaky column removed. Contract is honest.


## 5. Data limits

**Named Limitation: Survivorship Bias / New Content Gap.** 
Our current contract requires `clicks_feb > 5` to establish a baseline for decline. This deliberately excludes brand-new content created in late February or March. We cannot score the "refresh opportunity" for content that hasn't lived long enough to show a trend, meaning our model will have a blind spot for the first ~30 days of a page's life.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.